In [9]:
import os
import torch
from torch.utils.data import DataLoader

from dataset import (
    MaridaDatasetLoader,
    find_patch_bases,
    do_img_conf_mask_exist,
)
from preprocessing import (
    normalize_image,
    set_low_conf_for_nan,
    apply_augmentations,
    build_conf_ignore_mask,
    apply_ignore_index_to_target,
    flatten_for_rf,
    compute_dataset_stats,
)

In [10]:
def fix_base_name(base):
    """
    Prend '1-12-19_48MYU_0'
    et retourne 'S2_1-12-19_48MYU/S2_1-12-19_48MYU_0'
    """
    # Exemple : base = "1-12-19_48MYU_0"
    tile = base.rsplit("_", 1)[0]      # → "1-12-19_48MYU"
    folder = "S2_" + tile              # → "S2_1-12-19_48MYU"
    full = f"{folder}/S2_{base}"       # → "S2_1-12-19_48MYU/S2_1-12-19_48MYU_0"
    return full

In [11]:
########################
# 1. TRANSFORMS (CORRIGÉ)
########################

def train_transform(img, mask, conf):
    """
    Préprocessing appliqué pendant l'entraînement :
    - normalisation (Float + Division 10000)
    - NaN/Inf -> conf=3 + img nettoyée
    - augmentations géométriques
    """
    # 1. Gérer les NaN/Inf -> conf = 3, img nettoyée
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)

    # 2. Normalisation (Float) + DIVISION MANUELLE
    # preprocessing.normalize_image fait juste .float(), donc on divise ici
    img = normalize_image(img) / 10000.0

    # 3. Augmentations
    img, mask, conf = apply_augmentations(
        img,
        mask,
        conf,
        p_hflip=0.5,
        p_vflip=0.5,
        p_rotate90=0.5,
    )

    return img, mask, conf


def val_transform(img, mask, conf):
    """
    Préprocessing pour validation / test :
    - normalisation (Float + Division 10000)
    - NaN/Inf -> conf=3
    PAS d'augmentations.
    """
    # 1. Nettoyage
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    
    # 2. Normalisation + Division
    img = normalize_image(img) / 10000.0
    
    return img, mask, conf

In [12]:
########################
# 2. CONSTRUCTION DES DATASETS
########################

def build_bases(folder):
    """Retourne une liste de bases pour lesquelles img + mask + conf existent."""
    all_bases = find_patch_bases(folder)
    bases = [b for b in all_bases if do_img_conf_mask_exist(folder, b)]
    print(f"{folder} : {len(bases)} patches valides trouvés.")
    return bases

def load_split_list(split_file):
    """
    Lit un fichier de split (train / val / test)
    et retourne une liste de bases (strings) relatives à 'patches/'.
    Exemple de ligne dans le fichier : S2A_.../patch_0001
    """
    bases = []
    with open(split_file, "r") as f:
        for line in f:
            name = line.strip()
            if not name:
                continue
            # au cas où quelqu'un aurait mis .tif dans le fichier
            if name.endswith(".tif"):
                name = name[:-4]
            bases.append(name)
    return bases

def make_dataloaders(data_root, batch_size=4):
    """
    data_root = dossier 'raw' qui contient :
        - patches/
        - splits/ (train, val, test)
    """
    patches_root = os.path.join(data_root, "patches")
    splits_dir   = os.path.join(data_root, "splits")

    train_file = os.path.join(splits_dir, "train_X.txt")
    val_file   = os.path.join(splits_dir, "val_X.txt")

    # Charger les listes de bases depuis les fichiers
    train_bases = load_split_list(train_file)
    val_bases   = load_split_list(val_file)
    train_bases = [fix_base_name(b) for b in train_bases]
    val_bases   = [fix_base_name(b) for b in val_bases]

    # Vérifier que les fichiers .tif / _cl / _conf existent
    train_bases = [
        b for b in train_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]
    val_bases = [
        b for b in val_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]

    print(f"Train : {len(train_bases)} patches valides.")
    print(f"Val   : {len(val_bases)} patches valides.")

    train_dataset = MaridaDatasetLoader(
    folder=patches_root,
    bases=train_bases,
    transform=train_transform,
    )

    val_dataset = MaridaDatasetLoader(
    folder=patches_root,
    bases=val_bases,
    transform=val_transform,
    )


    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, val_loader

In [13]:
def make_dataloaders(data_root, batch_size=4):
    """
    data_root = dossier 'raw' (par ex: data/raw)
    Structure attendue :
        data_root/
            MARIDA/           <-- Nouveau sous-dossier
                patches/
                splits/
    """
    # --- MODIFICATION ICI ---
    # On ajoute le sous-dossier MARIDA au chemin racine
    marida_root = os.path.join(data_root, "MARIDA")
    
    patches_root = os.path.join(marida_root, "patches")
    splits_dir   = os.path.join(marida_root, "splits")
    # ------------------------

    train_file = os.path.join(splits_dir, "train_X.txt")
    val_file   = os.path.join(splits_dir, "val_X.txt")

    # Vérification rapide pour éviter les erreurs silencieuses
    if not os.path.exists(patches_root):
        raise FileNotFoundError(f"Dossier introuvable : {patches_root}")
    if not os.path.exists(train_file):
        raise FileNotFoundError(f"Fichier split introuvable : {train_file}")

    # Charger les listes de bases depuis les fichiers
    train_bases = load_split_list(train_file)
    val_bases   = load_split_list(val_file)
    
    # Correction des noms (S2_...)
    train_bases = [fix_base_name(b) for b in train_bases]
    val_bases   = [fix_base_name(b) for b in val_bases]

    # Vérifier que les fichiers .tif / _cl / _conf existent physiquement
    train_bases = [
        b for b in train_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]
    val_bases = [
        b for b in val_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]

    print(f"Train : {len(train_bases)} patches valides.")
    print(f"Val   : {len(val_bases)} patches valides.")

    train_dataset = MaridaDatasetLoader(
        folder=patches_root,
        bases=train_bases,
        transform=train_transform,
    )

    val_dataset = MaridaDatasetLoader(
        folder=patches_root,
        bases=val_bases,
        transform=val_transform,
    )

    # DataLoaders
    # num_workers=0 est plus sûr sous Windows/VS Code pour éviter les bugs
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0, 
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    return train_loader, val_loader

In [14]:
def make_test_loader(data_root, batch_size=4):
    """
    Crée le DataLoader pour le jeu de test.
    Utilise les mêmes corrections de chemin que pour le train/val.
    """
    # Construction des chemins
    marida_root = os.path.join(data_root, "MARIDA")
    patches_root = os.path.join(marida_root, "patches")
    splits_dir   = os.path.join(marida_root, "splits")
    test_file    = os.path.join(splits_dir, "test_X.txt")

    # Vérification de sécurité
    if not os.path.exists(test_file):
        print(f"⚠️ Attention : Fichier test introuvable ici : {test_file}")
        return None

    # 1. Chargement de la liste brute
    test_bases = load_split_list(test_file)
    
    # 2. Correction des noms (Ajout de 'S2_' et chemin complet)
    # On réutilise votre fonction fix_base_name définie plus haut
    test_bases = [fix_base_name(b) for b in test_bases]

    # 3. Filtrage : On ne garde que les fichiers qui existent vraiment
    test_bases = [
        b for b in test_bases
        if do_img_conf_mask_exist(patches_root, b)
    ]

    print(f"Test  : {len(test_bases)} patches valides.")

    # 4. Création du Dataset (Mode 'val_transform' car pas d'augmentation en test)
    test_dataset = MaridaDatasetLoader(
        folder=patches_root,
        bases=test_bases,
        transform=val_transform, 
    )

    # 5. Création du DataLoader
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False, # Pas besoin de mélanger pour le test
        num_workers=0, # Sécurité Windows
    )

    return test_loader

In [15]:
########################
# 3. EXEMPLE: CALCUL DES STATS
########################

def compute_stats_on_whole_dataset(data_root, batch_size=4):
    """
    Exemple de calcul de mean/std globales sur le dataset (sans augmentation).
    On utilise val_transform (sans aug) ou une transform spéciale si tu préfères.
    """
    train_folder = os.path.join(data_root, "train_X.txt")
    val_folder   = os.path.join(data_root, "val_X.txt")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    # Dataset sans augmentation (val_transform)
    full_dataset = torch.utils.data.ConcatDataset([
        MaridaDatasetLoader(folder=train_folder, bases=train_bases, transform=val_transform),
        MaridaDatasetLoader(folder=val_folder, bases=val_bases, transform=val_transform),
    ])

    full_loader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False)

    mean, std = compute_dataset_stats(full_loader)
    print("Mean per band:", mean)
    print("Std per band:", std)
    return mean, std

In [16]:
########################
# 4. EXEMPLE: BOUCLE D'ENTRAÎNEMENT (pseudo-code)
########################

def train_one_epoch(model, train_loader, optimizer, criterion, device="cpu"):
    model.train()

    for batch_idx, (imgs, masks, confs) in enumerate(train_loader):
        # imgs : (B, C, H, W)
        # masks: (B, H, W)
        # confs: (B, H, W)

        imgs  = imgs.to(device)
        masks = masks.to(device)
        confs = confs.to(device)

        # 1) Construire le target avec ignore_index basé sur la confidence
        targets_for_loss = []
        for b in range(imgs.shape[0]):
            conf_b = confs[b]   # (H, W)
            mask_b = masks[b]   # (H, W)

            ignore_mask = build_conf_ignore_mask(conf_b, threshold=2)
            target_mod  = apply_ignore_index_to_target(
                mask_b,
                ignore_mask,
                ignore_index=-100,
            )
            targets_for_loss.append(target_mod)

        targets_for_loss = torch.stack(targets_for_loss, dim=0)  # (B, H, W)

        # 2) Forward
        optimizer.zero_grad()
        logits = model(imgs)   # (B, num_classes, H, W) par exemple

        # 3) Loss (par ex. CrossEntropy2D avec ignore_index=-100)
        loss = criterion(logits, targets_for_loss)

        # 4) Backprop
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}, loss = {loss.item():.4f}")


########################
# 5. EXEMPLE: DATASET FLATTEN POUR RANDOM FOREST
########################

def build_rf_dataset(data_root):
    """
    Construit X, y pour RandomForest à partir de tous les patches
    (train + val, à adapter selon tes besoins).
    On utilise val_transform (pas d'augmentation aléatoire).
    """
    train_folder = os.path.join(data_root, "train")
    val_folder   = os.path.join(data_root, "val")

    train_bases = build_bases(train_folder)
    val_bases   = build_bases(val_folder)

    train_dataset = MaridaDatasetLoader(
        folder=train_folder,
        bases=train_bases,
        transform=val_transform,
    )
    val_dataset = MaridaDatasetLoader(
        folder=val_folder,
        bases=val_bases,
        transform=val_transform,
    )

    rf_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset])
    rf_loader  = DataLoader(rf_dataset, batch_size=1, shuffle=False)

    all_X = []
    all_y = []

    for img, mask, conf in rf_loader:
        img  = img.squeeze(0)   # (C, H, W)
        mask = mask.squeeze(0)  # (H, W)
        conf = conf.squeeze(0)  # (H, W)

        X_rf, y_rf = flatten_for_rf(img, mask, conf, conf_threshold=2)
        all_X.append(X_rf)
        all_y.append(y_rf)

    X_all = torch.cat(all_X, dim=0)
    Y_all = torch.cat(all_y, dim=0)

    print("RF dataset : X =", X_all.shape, ", y =", Y_all.shape)
    return X_all, Y_all

In [17]:

data_root = os.path.join(os.getcwd(), "..", "data", "raw")
data_root = os.path.abspath(data_root)
train_loader, val_loader = make_dataloaders(data_root, batch_size=4)
test_loader = make_test_loader(data_root, batch_size=4)


Train : 694 patches valides.
Val   : 328 patches valides.
Test  : 359 patches valides.


In [18]:
ds = train_loader.dataset

img1, mask1, conf1 = ds[0]
img2, mask2, conf2 = ds[0]

print("Same shape:", img1.shape, img2.shape)
print("Pixels exactly equal ?", torch.allclose(img1, img2))


Same shape: torch.Size([11, 256, 256]) torch.Size([11, 256, 256])
Pixels exactly equal ? False


In [22]:
import sys
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from unet import UNet

# --- 1. CONFIGURATION DES CHEMINS ---
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.append(os.path.join(project_root, "src"))

# Configuration des chemins de donnees
# On pointe vers data/raw/MARIDA car vos dossiers patches/splits sont dedans
DATA_ROOT = os.path.abspath(os.path.join(project_root, "data", "raw", "MARIDA"))
PATCHES_DIR = os.path.join(DATA_ROOT, "patches")
SPLITS_DIR = os.path.join(DATA_ROOT, "splits")

print(f"Dossier Patches vise : {PATCHES_DIR}")

# --- 2. IMPORTS ---
try:
    from dataset import MaridaDatasetLoader
    from unet import UNet
    from preprocessing import (
        normalize_image, set_low_conf_for_nan, apply_augmentations,
        build_conf_ignore_mask, apply_ignore_index_to_target
    )
except ImportError as e:
    print("ERREUR D'IMPORT : Verifiez vos fichiers dans le dossier src/")
    raise e

# --- 3. CHARGEMENT DES LISTES ---
def get_valid_bases_simple(split_filename, patches_root):
    split_path = os.path.join(SPLITS_DIR, split_filename)
    
    if not os.path.exists(split_path):
        print(f"Fichier manquant : {split_path}")
        return []
        
    valid_bases = []
    with open(split_path, 'r') as f:
        lines = f.readlines()
        
    print(f"Analyse de {split_filename}...")
    
    for line in lines:
        name = line.strip()
        if not name: continue
        
        # Correction : Ajout du prefixe S2_ manquant
        base_name = "S2_" + name   
        
        # Reconstruction du chemin vers le sous-dossier
        folder_name = base_name.rsplit('_', 1)[0]
        relative_path = os.path.join(folder_name, base_name)
        
        # Verification physique de l'existence du fichier
        if os.path.exists(os.path.join(patches_root, relative_path + ".tif")):
            valid_bases.append(relative_path)
            
    print(f"   -> {len(valid_bases)} images trouvees.")
    return valid_bases

# --- 4. PREPARATION ---
train_bases = get_valid_bases_simple("train_X.txt", PATCHES_DIR)
val_bases = get_valid_bases_simple("val_X.txt", PATCHES_DIR)

if len(train_bases) == 0:
    raise ValueError("Zero images trouvees. Verifiez le chemin DATA_ROOT.")

# --- 5. TRANSFORMS & LOADER ---
def train_transform(img, mask, conf):
    # Gestion des NaN
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    # Normalisation et division par 10000 pour mettre a l'echelle 0-1
    img = normalize_image(img) / 10000.0
    # Augmentations
    img, mask, conf = apply_augmentations(img, mask, conf)
    return img, mask, conf

def val_transform(img, mask, conf):
    img, conf = set_low_conf_for_nan(img, conf, low_conf_level=3)
    img = normalize_image(img) / 10000.0
    return img, mask, conf

train_dataset = MaridaDatasetLoader(PATCHES_DIR, train_bases, transform=train_transform)
val_dataset = MaridaDatasetLoader(PATCHES_DIR, val_bases, transform=val_transform)

BATCH_SIZE = 8
# Sur Windows avec VS Code, garder num_workers a 0 pour la stabilite
num_workers = 0 if os.name == 'nt' else 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(n_channels=11, n_classes=15).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
# On ignore la valeur -100 qui correspond aux pixels masques
criterion = nn.CrossEntropyLoss(ignore_index=-100)

# --- 6. ENTRAINEMENT ---
print(f"\nDemarrage de l'entrainement sur {DEVICE}...")
EPOCHS = 15

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for img, mask, conf in loop:
        img = img.to(DEVICE)
        mask = mask.to(DEVICE)
        conf = conf.to(DEVICE)
        
        # Application du masque de confiance (seuillage)
        ignore_mask = build_conf_ignore_mask(conf, threshold=2)
        target = apply_ignore_index_to_target(mask, ignore_mask, ignore_index=-100)
        
        # Decalage des classes de 1-15 vers 0-14
        valid = (target != -100)
        target[valid] = target[valid] - 1
        
        # Securite : si des pixels valaient 0 (fond), ils deviennent -1 -> on les ignore
        target[target == -1] = -100

        optimizer.zero_grad()
        outputs = model(img)
        loss = criterion(outputs, target)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    # Sauvegarde du modele
    model_dir = os.path.join(project_root, "models")
    os.makedirs(model_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(model_dir, "mon_unet_latest.pth"))

print("Entrainement termine avec succes.")

Dossier Patches vise : c:\Users\linya\OneDrive\Documents\GitHub\satellite_image_ocean_trash_detector\data\raw\MARIDA\patches
Analyse de train_X.txt...
   -> 694 images trouvees.
Analyse de val_X.txt...
   -> 328 images trouvees.

Demarrage de l'entrainement sur cpu...


Epoch 1/15:   0%|          | 0/87 [00:00<?, ?it/s]


KeyboardInterrupt: 